# Обучающий конвейер инструмента прогнозирования

Исследовательский ноутбук, реализующий §3.2 и §3.4 ВКР пошагово:

1. Загрузка / проверка биржевых данных (модуль 1).
2. Конструирование признаков и таргетов (модуль 2).
3. Обучение гибридной сети 1D-CNN + LSTM с MC Dropout (модуль 3).
4. **Сохранение артефакт-пакета** для бэкенда (`checkpoints/`).
5. MC Dropout инференс на test, оценка эпистемической неопределённости.
6. Двухступенчатый фильтр сигналов (модуль 3).
7. **Aggregate бэктест** на всех тикерах одновременно (модуль 4).
8. **Per-ticker бэктест** - отдельно по каждому активу.
9. Верификация против случайных портфелей по 3σ-критерию (§1.3, §3.4).

## 0. Подготовка окружения (Google Colab)

Если ноутбук запущен в Colab - клонируем публичный репозиторий
[lstm_exchange_predictor](https://github.com/Pozdnyakof/lstm_exchange_predictor)
и устанавливаем зависимости. В локальном Jupyter ячейка просто
определит корень проекта.

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} уже существует - пропускаем клонирование.')
    else:
        env = os.environ.copy()
        env['GIT_TERMINAL_PROMPT'] = '0'
        subprocess.run(
            [
                'git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                str(PROJECT_ROOT),
            ],
            check=True,
            env=env,
        )
        print(f'Репозиторий склонирован в {PROJECT_ROOT}')
else:
    # Локальный Jupyter: ноутбук в notebooks/, корень - папка выше.
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'IN_COLAB:     {IN_COLAB}')

In [ ]:
if IN_COLAB:
    print('Устанавливаем зависимости...')
    subprocess.run(
        [
            'pip', 'install', '--quiet',
            'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
            'scikit-learn>=1.4', 'requests>=2.31', 'yfinance>=0.2.40',
            'pydantic>=2.6', 'tqdm>=4.66', 'ipywidgets>=8.0',
        ],
        check=True,
    )
    print('Готово.')
else:
    print('Локальный режим: используются зависимости из текущего окружения.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s | %(message)s')

from graduate_work.config import default_config

cfg = default_config()
print(f'Тикеры ({len(cfg.data.tickers)}):', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)
print('MC проходов:', cfg.training.mc_passes)
print('Сигма-порог random monkeys:', cfg.trading.sigma_threshold)
print(f'Комиссии: {cfg.trading.commission_rate}, проскальзывание: {cfg.trading.slippage_rate}')

## 1. Подключение Google Drive и загрузка данных

Источник истины для биржевых данных - папка `MyDrive/lstm_exchange/data/raw/`,
наполненная ноутбуком `data_preparation.ipynb`. Здесь мы только
монтируем Drive, копируем данные в локальный `data/raw/` и при
необходимости докачиваем недостающее.

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    drive_data_raw = DRIVE_BASE / 'data' / 'raw'
    if drive_data_raw.exists():
        shutil.copytree(drive_data_raw, cfg.paths.data_raw, dirs_exist_ok=True)
        n_csv = sum(1 for _ in cfg.paths.data_raw.rglob('*.csv'))
        print(f'Подтянули с Drive {n_csv} CSV в {cfg.paths.data_raw}')
    else:
        print('На Drive пока нет данных - будем качать через download_all.')
        drive_data_raw.mkdir(parents=True, exist_ok=True)
else:
    print(f'Локальный режим, data/raw: {cfg.paths.data_raw}')

In [ ]:
from graduate_work.data.orchestrator import download_all

moex_dir = cfg.paths.data_raw / 'moex'
downloaded = sorted(p.stem for p in moex_dir.glob('*.csv')) if moex_dir.exists() else []
missing = [t for t in cfg.data.tickers if t not in downloaded]

if missing:
    print(f'Не хватает {len(missing)} тикеров - запускаем загрузку.')
    download_all(cfg.data, cfg.paths)
    if IN_COLAB:
        # Возвращаем только что скачанное на Drive, чтобы в следующий раз
        # подтянулось с него.
        shutil.copytree(cfg.paths.data_raw, DRIVE_BASE / 'data' / 'raw', dirs_exist_ok=True)
        print('Свежие файлы выгружены обратно на Drive.')
else:
    print(f'Все {len(downloaded)} тикеров уже на месте - пропускаем загрузку.')

## 2. Конструирование признаков и таргетов

SMA/EMA/RSI/моментум/волатильность + макроряды + индексные доходности.
Таргет - нормализованная лог-доходность по горизонтам [1, 5, 10, 20].

In [ ]:
from graduate_work.features import build_dataset

prepared = build_dataset(
    cfg.data, cfg.paths,
    persist=True,
    trading_cfg=cfg.trading,   # для cost-aware classification labels
)

print('Число фич:', prepared.num_features)
print('Целевые колонки:', prepared.target_cols)
print(f'Режим: {cfg.data.mode}')
for split_name in ('train', 'val', 'test'):
    arr = getattr(prepared, split_name)
    print(f'{split_name:>5s}:  X={arr["x"].shape}  y={arr["y"].shape}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(cfg.data.horizons), figsize=(14, 3), sharey=True)
for ax, hz, col in zip(axes, cfg.data.horizons, prepared.target_cols):
    values = prepared.train['y'][:, prepared.target_cols.index(col)]
    if cfg.data.mode == 'classification':
        # Бинарные метки {0, 1} (или сглаженные {eps, 1-eps}) -
        # показываем долю положительного класса.
        pos_share = float((values > 0.5).mean())
        ax.bar(['DOWN', 'UP'], [1.0 - pos_share, pos_share], color=['#dd8452', '#4c72b0'])
        ax.set_title(f'h={hz}, P(UP)={pos_share:.2f}')
        ax.set_ylim(0, 1)
    else:
        ax.hist(values, bins=80, color='#4c72b0', alpha=0.85)
        ax.set_title(f'h={hz}')
        ax.set_xlabel('норм. лог-доходность')
axes[0].set_ylabel('частота' if cfg.data.mode == 'regression' else 'доля')
fig.suptitle(f'Целевая переменная по горизонтам, mode={cfg.data.mode}')
fig.tight_layout(); plt.show()

## 3. Обучение модели

Гибридная сеть `causal Conv1D → LSTM → MLP`, Huber loss, early stopping.

In [ ]:
import torch

from graduate_work.model import ConvLstmRegressor
from graduate_work.training import Trainer, set_seed

set_seed(cfg.training.seed)

model = ConvLstmRegressor(
    input_dim=prepared.num_features,
    num_horizons=len(cfg.data.horizons),
    cfg=cfg.model,
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Параметров: {n_params:,}')
print('Устройство:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
trainer = Trainer(
    model, cfg.training,
    data_cfg=cfg.data,
    trading_cfg=cfg.trading,
)
history = trainer.fit(prepared.train, prepared.val)
print(f'Лучшая эпоха: {history.best_epoch}, val_loss={history.best_val_loss:.6f}')
print(f'Режим обучения: {cfg.data.mode}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
epochs = np.arange(1, len(history.train_loss) + 1)
ax.plot(epochs, history.train_loss, label='train', color='#4c72b0')
ax.plot(epochs, history.val_loss, label='val', color='#dd8452')
ax.axvline(history.best_epoch, color='#55a868', linestyle='--', label='best epoch')
ax.set_xlabel('эпоха'); ax.set_ylabel('Huber loss')
ax.set_title('Кривые обучения'); ax.legend()
fig.tight_layout(); plt.show()

## 4. Сохранение артефакт-пакета для бэкенда

После этого шага бэкенд готов работать в live-режиме: загрузит
веса, скейлер и метаинформацию из `checkpoints/`.

В Colab дополнительно копируем артефакты в Google Drive, чтобы их
можно было забрать локально.

In [ ]:
import dataclasses as _dc

from graduate_work.serving import ModelMeta, save_artifact
from graduate_work.serving.artifact import now_iso

meta = ModelMeta(
    feature_cols=list(prepared.feature_cols),
    target_cols=list(prepared.target_cols),
    horizons=list(cfg.data.horizons),
    window_size=cfg.data.window_size,
    num_features=prepared.num_features,
    num_horizons=len(cfg.data.horizons),
    model_config=_dc.asdict(cfg.model),
    training_date=now_iso(),
    tickers=list(cfg.data.tickers),
)
paths = save_artifact(model, prepared.scaler, meta, cfg.paths.checkpoints)
for k, p in paths.items():
    print(f'  {k:>8s}: {p}  ({p.stat().st_size/1024:.1f} KB)')

In [ ]:
# В Colab копируем артефакты в /MyDrive/lstm_exchange/checkpoints/
# (Drive уже смонтирован в шаге 1).
if IN_COLAB:
    drive_checkpoints = DRIVE_BASE / 'checkpoints'
    drive_checkpoints.mkdir(parents=True, exist_ok=True)
    for src in paths.values():
        shutil.copy2(src, drive_checkpoints / src.name)
    print(f'Артефакты сохранены в {drive_checkpoints}')
else:
    print(f'Локально артефакты лежат в {cfg.paths.checkpoints}')

## 5. MC Dropout инференс на test

50 стохастических проходов формируют распределение прогнозов;
среднее - точечная оценка, std - мера эпистемической неопределённости.

In [ ]:
from graduate_work.training import mc_predict

is_classification = cfg.data.mode == 'classification'

test = prepared.test
mean, std = mc_predict(
    model, test['x'],
    mc_passes=cfg.training.mc_passes,
    batch_size=cfg.training.batch_size,
    apply_sigmoid=is_classification,
)
print('mean shape:', mean.shape)
print('std shape :', std.shape)
print(f'Mode: {cfg.data.mode}')
if is_classification:
    print(f'Probabilities range: [{mean.min():.3f}, {mean.max():.3f}], mean={mean.mean():.3f}')
    print(f'Std range: [{std.min():.4f}, {std.max():.4f}], mean={std.mean():.4f}')
else:
    print(f'Средний std: {std.mean():.6f},  макс std: {std.max():.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for j, hz in enumerate(cfg.data.horizons):
    ax.scatter(mean[:, j], std[:, j], s=6, alpha=0.4, label=f'h={hz}')
ax.set_xlabel('MC mean (норм. лог-доходность)')
ax.set_ylabel('MC std (эпистемическая неопределённость)')
ax.set_title('Mean vs std MC Dropout по горизонтам (test)')
ax.legend(); fig.tight_layout(); plt.show()

## 6. Двухступенчатый фильтр и сигналы

In [ ]:
import dataclasses
import pandas as pd

from graduate_work.strategy import (
    SignalGenerator,
    attach_actual_targets,
    attach_lr_targets,
    build_predictions_frame,
    calibrate_bayes_threshold,
    calibrate_min_expected_return,
)

predictions = build_predictions_frame(
    timestamps=test['timestamp'],
    tickers=test['ticker'],
    mean=mean,
    std=std,
    horizons=cfg.data.horizons,
)

# Калибровка порога на val.
val = prepared.val
trading_cfg = cfg.trading
if val['x'].shape[0] > 0:
    val_mean, val_std = mc_predict(
        model, val['x'],
        mc_passes=cfg.training.mc_passes,
        batch_size=cfg.training.batch_size,
        apply_sigmoid=is_classification,
    )
    val_predictions = build_predictions_frame(
        timestamps=val['timestamp'],
        tickers=val['ticker'],
        mean=val_mean,
        std=val_std,
        horizons=cfg.data.horizons,
    )
    cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)
    if is_classification:
        lr_targets = attach_lr_targets(prepared.full_frame, val, cfg.data.horizons)
        calib = calibrate_bayes_threshold(
            val_predictions, lr_targets, cost_per_trade=cost_per_trade,
        )
        print(f'Bayes T={calib.min_expected_return:.4f}, val_signals={calib.n_val_signals}, '
              f'avg_actual={calib.val_avg_return:.5g}, wr={calib.val_win_rate:.3f}')
        trading_cfg = dataclasses.replace(cfg.trading, probability_threshold=calib.min_expected_return)
    else:
        val_targets = attach_actual_targets(val, cfg.data.horizons)
        calib = calibrate_min_expected_return(
            val_predictions, val_targets, cost_per_trade=cost_per_trade,
        )
        print(f'Calibrated T={calib.min_expected_return:.5g}, val_signals={calib.n_val_signals}, '
              f'avg={calib.val_avg_return:.5g}, wr={calib.val_win_rate:.3f}')
        trading_cfg = dataclasses.replace(cfg.trading, min_expected_return=calib.min_expected_return)

signals = SignalGenerator(trading_cfg, mode=cfg.data.mode).generate(predictions)
buys = signals[signals['action'] == 'BUY']
print(f'Всего записей сигналов: {len(signals)}')
print(f'BUY-сигналов: {len(buys)}')
if not buys.empty:
    print(f'Уникальных дат с BUY: {buys["timestamp"].nunique()}')
    print('Распределение BUY по тикерам (топ-10):')
    print(buys['ticker'].value_counts().head(10))

## 7. Aggregate бэктест

Прогон стратегии на всём тестовом периоде через единый событийный
движок. Капитал может одновременно держать до `max_positions`
тикеров, ребалансировка - по сигналам.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.backtest.engine import prices_from_full_frame

full = prepared.full_frame.copy()
if not isinstance(full.index, pd.DatetimeIndex):
    full.index = pd.to_datetime(full.index, utc=True)
test_start = pd.to_datetime(min(test['timestamp']), utc=True)
# Буфер на хвостовые позиции - max(horizons) баров (НЕ дней!).
buffer = cfg.data.bar_timedelta * max(cfg.data.horizons)
test_end = pd.to_datetime(max(test['timestamp']), utc=True) + buffer
test_prices = prices_from_full_frame(
    full.loc[(full.index >= test_start) & (full.index <= test_end)],
)

bt = run_backtest(signals, test_prices, trading_cfg)
bars_per_year = cfg.data.bars_per_year
metrics = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
print('=== AGGREGATE BACKTEST ===')
for k, v in metrics.items():
    print(f'{k:>20s}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
if not bt.equity.empty:
    ax.plot(bt.equity.index, bt.equity.values, color='#2ca02c', label='стратегия')
ax.axhline(cfg.trading.initial_capital, color='gray', linestyle=':', label='начальный капитал')
ax.set_title('Кривая капитала стратегии (aggregate)')
ax.set_xlabel('дата'); ax.set_ylabel('капитал, RUB')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 8. Per-ticker бэктест

Изолированный прогон по каждому тикеру: капитал делится поровну,
каждый актив получает собственный однотикерный движок. Это
показывает, на каких именно бумагах модель имеет предсказательное
преимущество.

In [ ]:
from graduate_work.backtest import run_per_ticker_backtest

per_ticker = run_per_ticker_backtest(
    signals, test_prices, trading_cfg,
    periods_per_year=bars_per_year,
)
per_ticker = per_ticker.sort_values('total_return', ascending=False).reset_index(drop=True)
print('=== PER-TICKER BACKTEST ===')
per_ticker

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sorted_pt = per_ticker.sort_values('total_return')
colors = ['#d62728' if r < 0 else '#2ca02c' for r in sorted_pt['total_return']]
ax.bar(sorted_pt['ticker'], sorted_pt['total_return'] * 100, color=colors)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('тикер'); ax.set_ylabel('итоговая доходность, %')
ax.set_title('Per-ticker: итоговая доходность стратегии')
plt.xticks(rotation=45)
fig.tight_layout(); plt.show()

n_profitable = int((per_ticker['total_return'] > 0).sum())
n_total = len(per_ticker)
print(f'Прибыльных тикеров: {n_profitable} из {n_total} ({100*n_profitable/n_total:.1f}%)')

## 9. Random monkeys и 3σ-критерий

Сравнение итога стратегии с распределением случайных портфелей,
соблюдающих те же инфраструктурные ограничения. Стратегия признаётся
статистически значимой при превышении порога mean + 3σ.

In [ ]:
from graduate_work.backtest import run_random_portfolios

avg_h = int(round(bt.trades['horizon'].mean())) if not bt.trades.empty else int(np.mean(cfg.data.horizons))
n_buy = int((signals['action'] == 'BUY').sum())
n_bars = int(len(test_prices.index.unique()))
trade_prob = max(min(n_buy / max(n_bars * trading_cfg.max_positions, 1), 1.0), 1e-4) if n_buy > 0 else 0.05
print(f'Random monkeys: avg_horizon={avg_h} bars, trade_prob={trade_prob:.4f}, BUY={n_buy}, bars={n_bars}')

report = run_random_portfolios(
    test_prices, trading_cfg,
    avg_horizon=avg_h,
    trade_probability=trade_prob,
    strategy_final=metrics['final_equity'],
    seed=cfg.training.seed,
)
print(f'Random monkeys: mean={report.mean:,.0f}  std={report.std:,.0f}')
print(f'3-sigma threshold: {report.threshold_value:,.0f}')
print(f'Strategy final:    {report.strategy_final:,.0f}')
print(f'Z-score:           {report.strategy_z_score:.3f}')
print(f'Significant:       {report.is_significant}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
returns_pct = report.final_returns * 100.0
ax.hist(returns_pct, bins=40, color='#9467bd', alpha=0.7, label='random monkeys')
strategy_ret = (report.strategy_final / cfg.trading.initial_capital - 1.0) * 100.0
threshold_ret = (report.threshold_value / cfg.trading.initial_capital - 1.0) * 100.0
ax.axvline(strategy_ret, color='#d62728', linewidth=2.5, label=f'стратегия ({strategy_ret:.2f}%)')
ax.axvline(threshold_ret, color='#2ca02c', linestyle='--', label=f'порог 3σ ({threshold_ret:.2f}%)')
ax.set_xlabel('итоговая доходность, %')
ax.set_ylabel('частота')
ax.set_title('Распределение random monkeys vs стратегия')
ax.legend(); fig.tight_layout(); plt.show()

## 10. Альтернативный фильтр: Split-Conformal (Vovk-Shafer)

Bayes-калиброванный порог опирается на «честные» вероятности модели.
Когда выход насыщен вокруг 0.5, формула `c_FP / (c_FP + c_FN)` упирается
в нижнюю планку 0.51, а std-фильтр блокирует все сигналы → 0 сделок.

**Conformal** настраивается под фактический output модели: вычисляет
квантиль `q = (1-α)·(n+1)/n` от residual scores `|prob - actual|` на val,
после чего пропускает сделку, если `prob > max(q, 1-q)`. В нашем R-0018
этот фильтр на тех же данных дал 1108 сделок и +85% return при том же
DSR=1.0 saturated, тогда как Bayes-порог давал 0 сделок.

Запускаем conformal **на тех же** test/val-предсказаниях, что и Bayes,
чтобы можно было сравнить metric-by-metric.


In [ ]:
# --- Conformal калибровка и сигналы ---
from graduate_work.strategy import ConformalSignalGenerator

if is_classification and val['x'].shape[0] > 0:
    val_targets_conf = attach_actual_targets(val, cfg.data.horizons)
    conf_gen = ConformalSignalGenerator(cfg.trading, alpha=0.1)
    conf_calib = conf_gen.calibrate(val_predictions, val_targets_conf)
    conf_signals = conf_gen.generate(predictions)
    print(f'Conformal: alpha=0.1, n_val={conf_calib.n_val_scores}, '
          f'q={conf_calib.quantile:.4f}, threshold={conf_calib.threshold:.4f}')
else:
    # В regression-режиме нужна правильная семантика actual; для краткости
    # пропускаем — conformal в этой работе позиционируется как фильтр для
    # классификационного output'а.
    conf_signals = pd.DataFrame(columns=signals.columns)
    print('Conformal флоу пропущен: cfg.data.mode != classification или val пуст.')

conf_buys = conf_signals[conf_signals['action'] == 'BUY']
print(f'BUY-сигналов (Conformal): {len(conf_buys)}')
if not conf_buys.empty:
    print(f'Уникальных дат с BUY: {conf_buys["timestamp"].nunique()}')
    print('Распределение BUY по тикерам (топ-10):')
    print(conf_buys['ticker'].value_counts().head(10))


In [ ]:
# --- Aggregate backtest: Conformal ---
if not conf_signals.empty:
    bt_conf = run_backtest(conf_signals, test_prices, cfg.trading)
    metrics_conf = compute_metrics(bt_conf.equity, bt_conf.trades, periods_per_year=bars_per_year)
else:
    bt_conf = None
    metrics_conf = {}

print('=== AGGREGATE BACKTEST: BAYES vs CONFORMAL ===')
all_keys = list(metrics.keys())
print(f'{"metric":>22s} | {"Bayes":>14s} | {"Conformal":>14s}')
print('-' * 58)
for k in all_keys:
    b = metrics.get(k, '—')
    c = metrics_conf.get(k, '—')
    fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
    print(f'{k:>22s} | {fmt(b):>14s} | {fmt(c):>14s}')


In [ ]:
# --- Equity curves: Bayes vs Conformal ---
fig, ax = plt.subplots(figsize=(10, 4))
if not bt.equity.empty:
    ax.plot(bt.equity.index, bt.equity.values, color='#1f77b4', label='Bayes filter')
if bt_conf is not None and not bt_conf.equity.empty:
    ax.plot(bt_conf.equity.index, bt_conf.equity.values, color='#ff7f0e', label='Conformal filter')
ax.axhline(cfg.trading.initial_capital, color='gray', linestyle=':', label='начальный капитал')
ax.set_title('Кривые капитала: Bayes vs Conformal')
ax.set_xlabel('дата'); ax.set_ylabel('капитал, RUB')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()


In [ ]:
# --- Random monkeys для Conformal стратегии ---
if bt_conf is not None and metrics_conf:
    avg_h_c = int(round(bt_conf.trades['horizon'].mean())) if not bt_conf.trades.empty else int(np.mean(cfg.data.horizons))
    n_buy_c = int((conf_signals['action'] == 'BUY').sum())
    n_bars_c = int(len(test_prices.index.unique()))
    trade_prob_c = max(min(n_buy_c / max(n_bars_c * cfg.trading.max_positions, 1), 1.0), 1e-4) if n_buy_c > 0 else 0.05
    print(f'Random monkeys (Conformal): avg_h={avg_h_c}, trade_prob={trade_prob_c:.4f}, BUY={n_buy_c}')

    report_conf = run_random_portfolios(
        test_prices, cfg.trading,
        avg_horizon=avg_h_c,
        trade_probability=trade_prob_c,
        strategy_final=metrics_conf['final_equity'],
        seed=cfg.training.seed,
    )
    print(f'Random monkeys: mean={report_conf.mean:,.0f}  std={report_conf.std:,.0f}')
    print(f'3-sigma threshold: {report_conf.threshold_value:,.0f}')
    print(f'Strategy final:    {report_conf.strategy_final:,.0f}')
    print(f'Z-score:           {report_conf.strategy_z_score:.3f}')
    print(f'Significant:       {report_conf.is_significant}')
else:
    report_conf = None
    print('Conformal backtest пуст — random monkeys не запускаем.')


In [ ]:
# --- Сравнительная сводка ---
print('=== ИТОГ: BAYES vs CONFORMAL ===')
print(f'{"":>22s} | {"Bayes":>14s} | {"Conformal":>14s}')
print('-' * 58)
n_buy_b = int((signals["action"] == "BUY").sum())
n_buy_c = int((conf_signals["action"] == "BUY").sum()) if not conf_signals.empty else 0
print(f'{"# BUY signals":>22s} | {n_buy_b:>14d} | {n_buy_c:>14d}')
print(f'{"final_equity":>22s} | {metrics.get("final_equity", 0):>14.0f} | {metrics_conf.get("final_equity", 0):>14.0f}')
if "total_return" in metrics:
    print(f'{"total_return":>22s} | {metrics["total_return"]:>14.4f} | {metrics_conf.get("total_return", 0):>14.4f}')
if "sharpe" in metrics:
    print(f'{"sharpe":>22s} | {metrics["sharpe"]:>14.4f} | {metrics_conf.get("sharpe", 0):>14.4f}')

z_b = report.strategy_z_score if 'report' in globals() else float('nan')
z_c = report_conf.strategy_z_score if report_conf is not None else float('nan')
print(f'{"z-score (3σ test)":>22s} | {z_b:>14.3f} | {z_c:>14.3f}')
sig_b = report.is_significant if 'report' in globals() else False
sig_c = report_conf.is_significant if report_conf is not None else False
print(f'{"significant":>22s} | {str(sig_b):>14s} | {str(sig_c):>14s}')


## Что дальше

- Артефакт-пакет лежит в `checkpoints/` (и в Google Drive, если
  блокнот запускался в Colab). Скопируйте 3 файла локально, чтобы
  бэкенд мог поднять live-режим.
- Запустите `scripts/05_run_server.py` локально: бэкенд загрузит
  модель, начнёт периодически опрашивать MOEX ISS, фронтенд
  покажет live-прогнозы и алерты по каждому тикеру.